# AgroInvest
## 02 - Análisis económico de inversiones

En este notebook se desarrolla el módulo económico-financiero de AgroInvest.

El objetivo es evaluar la viabilidad de distintas inversiones en tecnologías de Agricultura 4.0 a partir de:

- costes de implantación;
- superficie de la explotación;
- beneficio incremental esperado;
- vida útil de la inversión;
- tasa de descuento.

Se calcularán los principales indicadores de rentabilidad:

- ROI;
- Payback;
- VAN;
- TIR.

El análisis se realizará teniendo en cuenta la situación tecnológica actual de la explotación y el coste incremental necesario para incorporar nuevas tecnologías.

In [2]:
import pandas as pd
import numpy as np
import numpy_financial as npf
import matplotlib.pyplot as plt

In [81]:
df_constantes = pd.read_excel("Balance_economico.xlsx")

df_historicos = pd.read_excel(
    "Balance_economico_precios_distintos.xlsx"
)

df_inversiones = pd.read_excel(
    "gastos implantación tecnologia.xlsx"
)

In [83]:
for df in [df_constantes, df_historicos]:
    df.columns = df.columns.str.strip()
    df["Parcela"] = df["Parcela"].astype(str).str.strip()
    df["Cultivo"] = df["Cultivo"].astype(str).str.strip()

In [85]:
columnas_tecnologia = [
    "GPS",
    "RTK",
    "ISOBUS",
    "Siembra_variable",
    "Abono_variable",
    "Corte_tramos"
]

df_constantes["IIT"] = df_constantes[columnas_tecnologia].sum(axis=1)
df_historicos["IIT"] = df_historicos[columnas_tecnologia].sum(axis=1)

In [87]:
df_inversiones

,Unnamed: 0,euros por compra de cada tecnologia
0,GPS,11000
1,GPS+RTK,18000
2,ISOBUS+licencia,2500
3,Sembradora,35000
4,Abonadora,25000
5,Maquina fitosanitarios,62000


In [89]:
inversiones = {
    "GPS": 11000,
    "RTK_desde_GPS": 7000,
    "GPS_RTK_desde_cero": 18000,
    "ISOBUS": 2500,
    "Siembra_variable": 35000,
    "Abono_variable": 25000,
    "Corte_tramos": 62000
}

df_costes_tecnologia = pd.DataFrame(
    inversiones.items(),
    columns=["Tecnologia", "Coste_inversion"]
)

df_costes_tecnologia

,Tecnologia,Coste_inversion
0,GPS,11000
1,RTK_desde_GPS,7000
2,GPS_RTK_desde_cero,18000
3,ISOBUS,2500
4,Siembra_variable,35000
5,Abono_variable,25000
6,Corte_tramos,62000


In [91]:
#ROI
def calcular_roi(beneficio_anual, inversion):
    return (beneficio_anual / inversion) * 100

In [93]:
#Payback
def calcular_payback(beneficio_anual, inversion):
    if beneficio_anual <= 0:
        return np.inf
    
    return inversion / beneficio_anual

In [95]:
#VAN
def calcular_van(
    inversion,
    beneficio_anual,
    vida_util=15,
    tasa_descuento=0.05
):
    flujos = [-inversion] + [beneficio_anual] * vida_util
    return npf.npv(tasa_descuento, flujos)

In [97]:
#TIR
def calcular_tir(
    inversion,
    beneficio_anual,
    vida_util=15
):
    flujos = [-inversion] + [beneficio_anual] * vida_util
    return npf.irr(flujos) * 100

In [99]:
#prueba
inversion = 25000
beneficio_anual = 5000
vida_util = 15
tasa_descuento = 0.05

#calculo
roi = calcular_roi(
    beneficio_anual,
    inversion
)

payback = calcular_payback(
    beneficio_anual,
    inversion
)

van = calcular_van(
    inversion,
    beneficio_anual,
    vida_util,
    tasa_descuento
)

tir = calcular_tir(
    inversion,
    beneficio_anual,
    vida_util
)

print(f"ROI: {roi:.2f} %")
print(f"Payback: {payback:.2f} años")
print(f"VAN: {van:.2f} €")
print(f"TIR: {tir:.2f} %")

ROI: 20.00 %
Payback: 5.00 años
VAN: 26898.29 €
TIR: 18.42 %


In [105]:
# Costes de tecnologías

costes_tecnologia = {
    "GPS": 11000,
    "RTK": 7000,              # coste incremental si ya existe GPS
    "ISOBUS": 2500,
    "Siembra_variable": 35000,
    "Abono_variable": 25000,
    "Corte_tramos": 62000
}

In [107]:
def calcular_inversion_necesaria(tecnologias_actuales, tecnologias_objetivo):
    
    inversion_total = 0
    tecnologias_a_instalar = []

    for tecnologia in tecnologias_objetivo:
        
        if tecnologia not in tecnologias_actuales:
            
            inversion_total += costes_tecnologia[tecnologia]
            tecnologias_a_instalar.append(tecnologia)

    return inversion_total, tecnologias_a_instalar

In [111]:
combinaciones_tecnologicas = (
    df_constantes[
        [
            "Cultivo",
            "IIT",
            "GPS",
            "RTK",
            "ISOBUS",
            "Siembra_variable",
            "Abono_variable",
            "Corte_tramos"
        ]
    ]
    .drop_duplicates()
    .sort_values(["Cultivo", "IIT"])
)

combinaciones_tecnologicas

,Cultivo,IIT,GPS,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos
63,Arroz,1,1,0,0,0,0,0
67,Arroz,5,1,1,1,1,0,1
71,Arroz,5,1,1,1,1,1,0
74,Arroz,5,1,1,1,0,1,1
64,Arroz,6,1,1,1,1,1,1
0,Maiz,1,1,0,0,0,0,0
7,Maiz,4,1,1,1,0,1,0
3,Maiz,5,1,1,1,1,0,1
11,Maiz,5,1,1,1,0,1,1
1,Maiz,6,1,1,1,1,1,1


In [115]:
for cultivo in df_constantes["Cultivo"].unique():

    print(f"\n--- {cultivo} ---")

    display(
        combinaciones_tecnologicas[
            combinaciones_tecnologicas["Cultivo"] == cultivo
        ]
    )


--- Maiz ---


,Cultivo,IIT,GPS,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos
0,Maiz,1,1,0,0,0,0,0
7,Maiz,4,1,1,1,0,1,0
3,Maiz,5,1,1,1,1,0,1
11,Maiz,5,1,1,1,0,1,1
1,Maiz,6,1,1,1,1,1,1



--- Tomates ---


,Cultivo,IIT,GPS,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos
15,Tomates,1,1,0,0,0,0,0
2,Tomates,5,1,1,1,0,1,1



--- Arroz ---


,Cultivo,IIT,GPS,RTK,ISOBUS,Siembra_variable,Abono_variable,Corte_tramos
63,Arroz,1,1,0,0,0,0,0
67,Arroz,5,1,1,1,1,0,1
71,Arroz,5,1,1,1,1,1,0
74,Arroz,5,1,1,1,0,1,1
64,Arroz,6,1,1,1,1,1,1


In [117]:
#etiqueta de combinación
def crear_combinacion(row):
    tecnologias = []

    for tecnologia in columnas_tecnologia:
        if row[tecnologia] == 1:
            tecnologias.append(tecnologia)

    return " + ".join(tecnologias)

In [119]:
df_constantes["Combinacion_tecnologica"] = (
    df_constantes.apply(crear_combinacion, axis=1)
)

df_historicos["Combinacion_tecnologica"] = (
    df_historicos.apply(crear_combinacion, axis=1)
)

In [121]:
df_constantes[
    [
        "Cultivo",
        "IIT",
        "Combinacion_tecnologica"
    ]
].drop_duplicates().sort_values(["Cultivo", "IIT"])

,Cultivo,IIT,Combinacion_tecnologica
63,Arroz,1,GPS
67,Arroz,5,GPS + RTK + ISOBUS + Siembra_variable + Corte_...
71,Arroz,5,GPS + RTK + ISOBUS + Siembra_variable + Abono_...
74,Arroz,5,GPS + RTK + ISOBUS + Abono_variable + Corte_tr...
64,Arroz,6,GPS + RTK + ISOBUS + Siembra_variable + Abono_...
0,Maiz,1,GPS
7,Maiz,4,GPS + RTK + ISOBUS + Abono_variable
3,Maiz,5,GPS + RTK + ISOBUS + Siembra_variable + Corte_...
11,Maiz,5,GPS + RTK + ISOBUS + Abono_variable + Corte_tr...
1,Maiz,6,GPS + RTK + ISOBUS + Siembra_variable + Abono_...


In [123]:
resumen_combinaciones = (
    df_constantes
    .groupby(
        [
            "Cultivo",
            "IIT",
            "Combinacion_tecnologica"
        ]
    )
    .agg(
        Observaciones=("Beneficio_por_ha", "size"),
        Beneficio_medio_ha=("Beneficio_por_ha", "mean"),
        Rendimiento_medio=("Rend_t_ha", "mean"),
        Consumo_medio=("Consumo_L_ha", "mean")
    )
    .round(2)
    .reset_index()
)

resumen_combinaciones

,Cultivo,IIT,Combinacion_tecnologica,Observaciones,Beneficio_medio_ha,Rendimiento_medio,Consumo_medio
0,Arroz,1,GPS,7,-260.48,7.20,20.00
1,Arroz,5,GPS + RTK + ISOBUS + Abono_variable + Corte_tr...,6,148.17,8.14,16.92
2,Arroz,5,GPS + RTK + ISOBUS + Siembra_variable + Abono_...,6,132.03,8.06,16.92
3,Arroz,5,GPS + RTK + ISOBUS + Siembra_variable + Corte_...,3,289.20,8.47,16.92
4,Arroz,6,GPS + RTK + ISOBUS + Siembra_variable + Abono_...,20,338.32,8.53,16.61
5,Maiz,1,GPS,9,1011.53,13.43,12.50
6,Maiz,4,GPS + RTK + ISOBUS + Abono_variable,2,1286.10,14.42,11.14
7,Maiz,5,GPS + RTK + ISOBUS + Abono_variable + Corte_tr...,7,1691.28,16.02,10.54
8,Maiz,5,GPS + RTK + ISOBUS + Siembra_variable + Corte_...,3,1564.22,15.65,10.54
9,Maiz,6,GPS + RTK + ISOBUS + Siembra_variable + Abono_...,30,2010.25,16.96,10.20


## Conclusión

En este notebook se han definido y validado las principales funciones financieras que utilizará AgroInvest para evaluar inversiones tecnológicas: ROI, Payback, VAN y TIR.

También se han estructurado los costes de implantación de las distintas tecnologías y se ha desarrollado una función para calcular la inversión necesaria en función de las tecnologías ya disponibles en la explotación.

El análisis de las combinaciones tecnológicas históricas permite conocer qué configuraciones se han observado en los datos disponibles. Sin embargo, estas combinaciones no representan todas las configuraciones técnicamente posibles y, por tanto, no se utilizarán como un catálogo cerrado de alternativas.

La estimación del beneficio incremental asociado a nuevas combinaciones tecnológicas se abordará posteriormente mediante un modelo predictivo, que alimentará el motor financiero desarrollado en este notebook.